# TEEP Proctoring Services (cascade) — VLM cascade + CV, combined Kaggle runtime

Same shape as `teep_proctoring_services.ipynb`, but the screen side runs the
**two-model cascade** instead of the single Qwen2.5-VL-3B service:

- `vlm_service_cascade.py` — SmolVLM-500M screens, Qwen2.5-VL decides on
  escalation (port `8791`)
- `cv_service.py` — head/gaze + YOLO + review-score pipeline (port `8789`)

```text
screenshot -> SmolVLM-500M -> [confident AND nothing suspicious?]
                                 |- yes -> not cheating       (fast path, no large-model call)
                                 '- no  -> Qwen2.5-VL + 2nd opinion -> verdict
```

Each service is exposed through its own `cloudflared` quick tunnel. The two
resulting URLs are what you set as `VLM_URL` / `CV_URL` when starting
`server.js` on your own machine.

## Before running

1. **Settings (right sidebar) → Accelerator → GPU T4 x2**, and **Internet → On**.
   Two GPUs matter more here than in the single-model notebook: three models
   now share the session.
2. Attach the same Kaggle Dataset with the CV assets (next cell for the exact
   layout) — **Add Input** in the right sidebar.
3. Run top to bottom. Re-run just the tunnel cells after a restart — the URLs
   change every time.

## Which large model to use here

This notebook defaults `LARGE_MODEL_ID` to **Qwen2.5-VL-3B**, not 7B. On T4 x2
the 7B in fp16 is ~15 GB of weights and does not fit in a single T4 alongside
activations, so it has to be sharded across both GPUs — which then competes
with SmolVLM and the whole CV stack (YOLO + L2CS + MediaPipe) for the same
memory.

- **Live end-to-end test** (this notebook, all three services): use the 3B.
- **7B-vs-cascade comparison numbers**: use `teep_cascade_vlm_service.ipynb`
  instead, where the VLM has both GPUs to itself. Section 5 below shows the
  exact settings if you still want to try 7B here.

## 0. Check the GPUs actually attached

In [ ]:
!nvidia-smi


## 1. Attach the CV model + source Dataset

Prepare this **once**, locally, from files already in the repo, then upload
as a private Kaggle Dataset (Kaggle UI → Create → New Dataset) and attach it
to this notebook:

```text
teep-kaggle-bundle/
├── integration/                       # computer_vision/src/integration/*.py
│   ├── 01_head_gaze_adapter.py
│   ├── 02_yolo_output_adapter.py
│   ├── 03_event_manager.py
│   └── 05_multi_cue_review_score.py
├── configs/
│   └── multi_cue_review_score_v1_1.json    # computer_vision/configs/...
├── resources/
│   └── mediapipe/
│       └── mediapipe_expanded_subset_14.csv    # computer_vision/resources/mediapipe/...
└── models/                            # flat — no yolo/l2cs/mediapipe subfolders
    ├── original_5e_best.pt
    ├── L2CSNet_gaze360.pkl
    └── face_landmarker.task
```

The three files under `models/` come from the project's Google Drive folder
(see `computer_vision/models/README.md` for the download link + checksums to
verify against). Everything else is copied straight from this repo's
`computer_vision/` folder — no rewriting needed, `computer_vision/` is not
touched by this notebook, only read from.

If your dataset's slug differs from `teep-kaggle-bundle`, change
`DATASET_SLUG` below to match.


In [ ]:
import os
from pathlib import Path

DATASET_SLUG = "teep-kaggle-bundle"  # change if your dataset's slug differs
DATASET_DIR = Path("/kaggle/input") / DATASET_SLUG

required = [
    DATASET_DIR / "integration" / "01_head_gaze_adapter.py",
    DATASET_DIR / "integration" / "02_yolo_output_adapter.py",
    DATASET_DIR / "integration" / "03_event_manager.py",
    DATASET_DIR / "integration" / "05_multi_cue_review_score.py",
    DATASET_DIR / "configs" / "multi_cue_review_score_v1_1.json",
    DATASET_DIR / "resources" / "mediapipe" / "mediapipe_expanded_subset_14.csv",
    DATASET_DIR / "models" / "original_5e_best.pt",
    DATASET_DIR / "models" / "L2CSNet_gaze360.pkl",
    DATASET_DIR / "models" / "face_landmarker.task",
]

missing = [str(p) for p in required if not p.exists()]
if missing:
    print("Missing from the attached dataset (fix the dataset or DATASET_SLUG above):")
    for m in missing:
        print("  -", m)
    raise SystemExit("Dataset check failed — see missing paths above.")

print("Dataset OK:", DATASET_DIR)


## 2. Install dependencies

Kaggle's GPU image already ships a working CUDA-linked `torch`/`torchvision`
— **don't** force-reinstall them, that risks breaking the preinstalled CUDA
wiring. This only installs what's missing for the two services.


In [ ]:
!pip install -q flask pillow accelerate "transformers>=4.49" qwen-vl-utils
!pip install -q --no-deps l2cs==0.0.1
# l2cs/pipeline.py does `from face_detection import RetinaFace`, a separate
# package NOT pulled in by l2cs's own deps (and not the same as `pip install
# face_detection` from PyPI, which is a different unrelated package) --
# without this, HeadGazeAdapter.start() fails with
# "ModuleNotFoundError: No module named 'face_detection'" on every /frame call.
# --no-deps here too: this package's own setup.py pulls in an unpinned torch,
# which can silently replace Kaggle's preinstalled CUDA-linked torch build
# with a CPU-only one -- the actual cause if a later step reports
# torch.cuda.is_available()==False despite the GPU accelerator being on.
!pip install -q --no-deps git+https://github.com/elliottzheng/face-detection.git@master
!pip install -q ultralytics==8.4.86 mediapipe==0.10.35 opencv-python-headless==5.0.0.93


In [ ]:
import torch
assert torch.cuda.is_available(), (
    "torch.cuda.is_available() is False after the installs above -- STOP "
    "here and fix this before continuing, or every /frame request will fail "
    "deep in inference instead of here. Check: (1) Settings > Accelerator is "
    "GPU, not None; (2) restart the kernel and re-run from the top if a pip "
    "install above silently replaced the preinstalled CUDA torch build."
)
print("torch.cuda.is_available():", torch.cuda.is_available())
print("device:", torch.cuda.get_device_name(0))


## 3. Clone L2CS-Net (public source, no auth needed)

`cv_service.py` imports from this checked-out directory via `sys.path`, not
from the `l2cs` pip package (see `computer_vision/models/README.md`).


In [ ]:
!git clone --depth 1 https://github.com/Ahmednull/L2CS-Net /kaggle/working/L2CS-Net


## 4. Write out the two services

Embedded verbatim from `exam-monitor-extension/vlm_service_cascade.py` and
`exam-monitor-extension/cv_service.py` in the repo — keep these two cells in
sync if you edit the source files.

In [ ]:
%%writefile /kaggle/working/vlm_service_cascade.py
"""Cascaded two-model VLM analysis service for the exam monitor.

Implements the architecture in Fig. "cascade-flow": a small, cheap model
(SmolVLM-500M) screens every screenshot; only screenshots it cannot clear
confidently are escalated to the large, dominant model (Qwen2.5-VL), which
receives the small model's assessment as an advisory second opinion
(single-round consultation, NOT an iterative debate).

    screenshot -> SmolVLM-500M -> [confident AND plainly just the exam page?]
                                     |- yes -> not cheating          (fast path)
                                     '- no  -> Qwen2.5-VL + 2nd opinion -> verdict

The gate asks "is this plainly nothing but the exam page", NOT "did the small
model decide it is not cheating". Those differ whenever the small model returns
an unusable answer or misjudges exam_relevance, and reading either as innocence
made the cascade skip exactly the screenshots it exists to catch.

Why cascade and not a debate loop: an iterative exchange multiplies
T_inference by the number of rounds, and the 500M model is far too weak to
change a 7B model's mind through argument. The cascade instead SAVES
inference on the majority of benign screenshots, which is what the
end-to-end latency metric (T_total = T_capture + T_network + T_inference)
actually rewards.

The same service also runs the two single-model baselines, so an accuracy or
latency difference cannot come from a different code path, prompt or parser:

    MODE=cascade     small screens, large decides on escalation   (default)
    MODE=large_only  large model on every screenshot              (baseline A)
    MODE=small_only  small model on every screenshot              (baseline B)

Endpoints:
  GET  /health                        -> models, devices, mode, threshold
  GET  /config                        -> {lang, exam_context, mode, conf_threshold}
  POST /config  {"lang": "en"}        -> switch to a built-in exam context
                {"exam_context": "..."} -> set a custom one
                {"mode": "large_only"}  -> switch routing mode
                {"conf_threshold": 0.6} -> switch the escalation gate
  POST /analyze {"image": "<dataURL|base64>"}
       -> {category, identity, exam_relevance, summary, is_cheating,
           latency_ms, path, escalated, small, large}
  GET  /stats                         -> escalation rate + per-path latency
  POST /stats/reset                   -> clear the counters between runs

Setup:
  pip install flask torch transformers qwen-vl-utils accelerate pillow

Run (alongside the single-model service on 8788):
  REQUIRE_CUDA=1 PORT=8791 python vlm_service_cascade.py

VRAM: Qwen2.5-VL-7B in fp16 is ~15 GB of weights and does NOT fit in one T4's
usable ~15 GB once activations are added. Placement options:

  LARGE_DEVICE=auto            shard the large model across every visible GPU
                               (the only way to run 7B in fp16 on T4 x2)
  LARGE_MAX_MEMORY=0:9GiB,1:9GiB   cap the shard per GPU, so the small model
                               and cv_service.py still have room
  LARGE_LOAD_4BIT=1            ~5 GB instead, fits one GPU (needs bitsandbytes,
                               and changes the numerics you are reporting)
  LARGE_MODEL_ID=Qwen/Qwen2.5-VL-3B-Instruct   what the single-model deployment
                               already uses; ~7 GB, comfortable on one T4

Defaults (no env set): large on cuda:0, small on cuda:1 when a second GPU
exists, otherwise both on the same device.
"""
import base64
import io
import json
import os
import re
import threading
import time
from datetime import datetime

# Reduce fragmentation-driven OOM (must be set before CUDA context init).
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

from flask import Flask, request, jsonify
from PIL import Image
import torch
import transformers
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration
from qwen_vl_utils import process_vision_info

# transformers 5.x dropped AutoModelForVision2Seq in favour of
# AutoModelForImageTextToText (Kaggle images already ship 5.x); keep the old
# name working so the same file runs on an older local install too.
try:
    from transformers import AutoModelForImageTextToText as AutoVLM
except ImportError:                                     # transformers < 4.46
    from transformers import AutoModelForVision2Seq as AutoVLM

# Same story for the dtype kwarg: renamed torch_dtype -> dtype in 5.x. The old
# name is still accepted there "for BC", but that BC will not last forever.
_DTYPE_KW = "dtype" if int(transformers.__version__.split(".")[0]) >= 5 else "torch_dtype"

# ---------------------------------------------------------------- config ----

LARGE_MODEL_ID = os.environ.get("LARGE_MODEL_ID", "Qwen/Qwen2.5-VL-7B-Instruct")
SMALL_MODEL_ID = os.environ.get("SMALL_MODEL_ID", "HuggingFaceTB/SmolVLM-500M-Instruct")

PORT = int(os.environ.get("PORT", 8791))
HOST = os.environ.get("HOST", "127.0.0.1")

MODE = os.environ.get("MODE", "cascade")            # cascade|large_only|small_only
VALID_MODES = {"cascade", "large_only", "small_only"}

# Gate for the fast path. The small model must report a usable answer of
# exactly "exam_page" AND be confident above this threshold; anything else
# escalates. Raising it escalates more often -> safer, slower. This is the knob
# to sweep when reporting the accuracy/latency trade-off.
CONF_THRESHOLD = float(os.environ.get("CONF_THRESHOLD", 0.60))

# Visual-token caps. Keep identical to the benchmark notebook, or the accuracy
# you report is not the accuracy you deployed.
MIN_PIXELS = 256 * 28 * 28
MAX_PIXELS = 1024 * 28 * 28
SMALL_LONGEST_EDGE = int(os.environ.get("SMALL_LONGEST_EDGE", 4 * 384))

MAX_NEW_TOKENS = 192
SMALL_MAX_NEW_TOKENS = 160

# Categories that can constitute cheating. exam_page is excluded: an official
# exam-platform page is never cheating regardless of relevance.
CHEAT_CATEGORIES = {"ai_assistant", "search_engine",
                    "reference_material", "other_application"}

# Every label the prompt allows. Anything outside this set means the model did
# not actually answer -- most often a small model echoing the prompt's JSON
# template verbatim ("<one label above>"), which is syntactically valid JSON and
# therefore sails through the parser. Treated as an invalid answer, never as
# "nothing suspicious".
VALID_CATEGORIES = CHEAT_CATEGORIES | {"exam_page"}
VALID_RELEVANCE = {"direct", "partial", "none"}

EXAM_CONTEXTS = {
    "en": ("An online English level test (International English Test), 60 multiple-choice "
           "questions on English grammar and vocabulary. Current question (Q3, grammar): "
           "'____ 20 chairs in the office.' Options: 'This is', 'There is', 'They are', "
           "'There are'."),
    "ko": ("An online Korean language level test (90 Day Korean Level Test), multiple-choice "
           "questions on basic Korean vocabulary and expressions. Current question (Q2): "
           "'How do you say \"thank you\" in Korean?' Options: 죄송합니다, 사랑해요, 감사합니다, "
           "안녕하세요."),
    "zh": ("An online Chinese proficiency practice test (Dong Chinese, HSK level 1), matching "
           "a sentence to the correct picture. Current question: '这是我先生的衣服。' — choose "
           "the image that matches the text."),
    "ja": ("An online Japanese proficiency practice test (JLPTCheck, JLPT style), "
           "fill-in-the-blank multiple choice. Current question: "
           "'きょうは５００＿およぎました。' Options: ど, ばん, メートル, グラム."),
}

LANG = os.environ.get("EXAM_LANG", "en")
EXAM_CONTEXT = EXAM_CONTEXTS.get(LANG, EXAM_CONTEXTS["en"])

if MODE not in VALID_MODES:
    raise SystemExit("MODE must be one of %s, got %r" % (sorted(VALID_MODES), MODE))

if os.environ.get("REQUIRE_CUDA") == "1":
    assert torch.cuda.is_available(), (
        "REQUIRE_CUDA=1 but no GPU is visible. On Kaggle: "
        "Settings > Accelerator > GPU T4 x2, then restart the session.")

# Serialize generate() across both models: a burst of screenshots must never
# run two forward passes at once, which would multiply peak VRAM.
_GEN_LOCK = threading.Lock()
_STATS_LOCK = threading.Lock()

# ---------------------------------------------------------------- prompts ---

# NOTE: contains literal { } for the JSON schema -> fill with .replace(),
# never .format() / f-string.
PROMPT_TEMPLATE = """You are a screen-analysis component in an online-exam monitoring system.

EXAM CONTEXT (what the student is supposed to be working on):
{EXAM_CONTEXT}

Given ONE screenshot of a student's screen, produce TWO independent judgments.

1) category - classify the PRIMARY non-exam content into exactly ONE label:
- exam_page: the online exam or quiz itself, or an official page of the exam platform (instructions, an allowed calculator)
- ai_assistant: an AI chatbot (ChatGPT, Claude, Gemini, Copilot, Perplexity, Doubao, Kimi, ERNIE/Wenxin, DeepSeek, etc.)
- search_engine: a search engine or its results page (Google, Bing, DuckDuckGo, Baidu, Naver, Yahoo! Japan)
- reference_material: encyclopedia, dictionary, docs, tutorial, PDF, or Q&A site (Wikipedia, Namu Wiki, Baidu Baike, Zdic, MDN, StackOverflow, Zhihu)
- other_application: any other website or app (news, video, shopping, maps, social, messaging like WeChat/LINE, music, a notes or word-processor window, etc.)

MULTI-WINDOW RULE: if the exam is visible at the same time as another resource
(split screen, two windows side by side, a popup over the exam), classify the
OTHER resource, not the exam. Choose exam_page ONLY when the exam or an official
exam-platform page is the only substantive content on screen.

2) exam_relevance - how related the visible content is to the EXAM CONTEXT above:
- direct: the exam's question, its wording, or its answer is visibly being looked up, typed, or answered
- partial: the same subject or topic as the exam, but not the specific question
- none: unrelated to the exam content - a different subject, entertainment, personal matters, or a neutral/empty page

Judge exam_relevance ONLY against the EXAM CONTEXT above. A page can be academic
and still be "none" if it has nothing to do with this exam. If category is
exam_page, set exam_relevance to "direct".

The screen may be in ANY language (including Chinese, Japanese, Korean). Classify by
meaning and visual layout, NOT by language. Always write "summary" in English.

Output ONLY valid JSON (no markdown):
{"category":"<one label above>","identity":"<site/app name if legible else 'unknown'>","exam_relevance":"<direct|partial|none>","summary":"<one English sentence describing what is shown>"}"""

# Appended to the large model's prompt on escalation. Advisory, never binding:
# the dominant model must be free to overrule a wrong screening verdict,
# otherwise the cascade would inherit the small model's errors.
SECOND_OPINION_BLOCK = """

SECOND OPINION (from a smaller screening model that flagged this screenshot for review):
  category: {SO_CATEGORY}
  exam_relevance: {SO_RELEVANCE}
  identity: {SO_IDENTITY}
  summary: {SO_SUMMARY}
  confidence: {SO_CONF}

Treat this as advisory only. It comes from a weaker model and may be wrong.
Judge the screenshot yourself and overrule it whenever your own reading differs."""


def build_prompt():
    return PROMPT_TEMPLATE.replace("{EXAM_CONTEXT}", EXAM_CONTEXT)


def build_escalation_prompt(small_out, small_conf):
    block = (SECOND_OPINION_BLOCK
             .replace("{SO_CATEGORY}", str(small_out.get("category", "unclear")))
             .replace("{SO_RELEVANCE}", str(small_out.get("exam_relevance", "none")))
             .replace("{SO_IDENTITY}", str(small_out.get("identity", "unknown")))
             .replace("{SO_SUMMARY}", str(small_out.get("summary", ""))[:200])
             .replace("{SO_CONF}", "%.2f" % small_conf))
    return build_prompt() + block


# ------------------------------------------------------------- utilities ----

def dataurl_to_pil(s):
    if isinstance(s, str) and s.strip().startswith("data:") and "," in s:
        s = s.split(",", 1)[1]
    return Image.open(io.BytesIO(base64.b64decode(s))).convert("RGB")


def parse_json(raw):
    """Return (dict, parsed_ok). A parse failure is a hard escalation trigger."""
    s = re.sub(r"```$", "", re.sub(r"^```(?:json)?", "", raw.strip())).strip()
    for cand in (s, None):
        if cand is None:
            m = re.search(r"\{.*\}", s, re.DOTALL)
            if not m:
                break
            cand = m.group(0)
        try:
            d = json.loads(cand)
            if isinstance(d, dict):
                return d, True
        except Exception:
            continue
    return ({"category": "unclear", "identity": "unknown",
             "exam_relevance": "none", "summary": raw[:160]}, False)


def decide(out):
    """Apply the cheating rule. Returns (is_cheating, warning_or_None).

    A missing exam_relevance is treated as "none" so the service keeps serving,
    but it is surfaced -- silently defaulting to "none" makes the system flag
    nothing at all, which is indistinguishable from a well-behaved student.
    """
    cat = str(out.get("category") or "").strip().lower()
    rel_raw = out.get("exam_relevance")
    warning = None
    if rel_raw is None:
        warning = "model did not return exam_relevance; treated as 'none'"
        rel = "none"
    else:
        rel = str(rel_raw).strip().lower()
        if rel not in {"direct", "partial", "none"}:
            warning = "unexpected exam_relevance %r; treated as 'none'" % (rel_raw,)
            rel = "none"
    return (cat in CHEAT_CATEGORIES and rel != "none"), warning


def output_valid(out):
    """Did the model actually answer? Returns (ok, reason_or_None).

    Separate from decide() on purpose. decide() answers "is this cheating",
    which for any unrecognised label is False -- and a False from garbage must
    never be read as evidence of innocence. This tells the gate whether the
    answer is usable at all.
    """
    cat = str(out.get("category") or "").strip().lower()
    rel = str(out.get("exam_relevance") or "").strip().lower()
    if cat not in VALID_CATEGORIES:
        return False, "category %r is not one of the allowed labels" % (
            out.get("category"),)
    if rel not in VALID_RELEVANCE:
        return False, "exam_relevance %r is not one of the allowed values" % (
            out.get("exam_relevance"),)
    return True, None


def sequence_confidence(model, sequences, scores):
    """Mean per-token probability of the greedy generation, in [0, 1].

    Used as the small model's confidence. Not a calibrated probability of
    being correct -- it is a proxy -- but it is monotonic enough to gate on,
    and it is computed identically for every screenshot, which is what the
    escalation-rate comparison needs.
    """
    try:
        tr = model.compute_transition_scores(sequences, scores, normalize_logits=True)
        tr = tr.float()
        mask = torch.isfinite(tr)
        if mask.sum() == 0:
            return 0.0
        return float(torch.exp(tr[mask].mean()).clamp(0.0, 1.0))
    except Exception:
        return 0.0


# ------------------------------------------------------------- model defs ---

DTYPE = torch.float16 if torch.cuda.is_available() else torch.float32
N_GPU = torch.cuda.device_count() if torch.cuda.is_available() else 0

# Default placement: large on cuda:0, small on the second GPU when there is
# one (Kaggle T4 x2), otherwise both on the same device.
#
# LARGE_DEVICE=auto shards the large model across every visible GPU via
# accelerate. Needed for Qwen2.5-VL-7B in fp16 (~15 GB of weights), which does
# NOT fit in one T4's usable ~15 GB once activations are added -- pinning it to
# a single device OOMs during the first forward pass, not at load time.
LARGE_DEVICE = os.environ.get("LARGE_DEVICE") or ("cuda:0" if N_GPU else "cpu")
LARGE_SHARDED = LARGE_DEVICE.strip().lower() == "auto"
SMALL_DEVICE = os.environ.get("SMALL_DEVICE") or (
    "cuda:1" if N_GPU > 1 else ("cuda:0" if N_GPU else "cpu"))


class QwenVLM:
    """The dominant model. float16 + sdpa: a T4 is Turing (sm75), which has
    neither native bf16 nor flash-attention-2."""

    def __init__(self):
        print("Loading LARGE", LARGE_MODEL_ID, "->", LARGE_DEVICE, "...", flush=True)
        kw = {_DTYPE_KW: DTYPE, "attn_implementation": "sdpa"}
        if os.environ.get("LARGE_LOAD_4BIT") == "1":
            from transformers import BitsAndBytesConfig
            kw["quantization_config"] = BitsAndBytesConfig(
                load_in_4bit=True, bnb_4bit_compute_dtype=DTYPE,
                bnb_4bit_quant_type="nf4")
            kw["device_map"] = "auto" if LARGE_SHARDED else {"": LARGE_DEVICE}
        elif torch.cuda.is_available():
            kw["device_map"] = "auto" if LARGE_SHARDED else {"": LARGE_DEVICE}
            if LARGE_SHARDED and os.environ.get("LARGE_MAX_MEMORY"):
                # e.g. LARGE_MAX_MEMORY="0:9GiB,1:9GiB" -- leaves room on both
                # GPUs for the small model and the CV service, instead of
                # letting accelerate fill cuda:0 to the brim first.
                kw["max_memory"] = dict(
                    (int(k), v) for k, v in
                    (part.split(":", 1) for part in
                     os.environ["LARGE_MAX_MEMORY"].split(",")))
        self.model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
            LARGE_MODEL_ID, **kw)
        if not torch.cuda.is_available():
            self.model.to("cpu")
        self.model.eval()
        self.processor = AutoProcessor.from_pretrained(
            LARGE_MODEL_ID, min_pixels=MIN_PIXELS, max_pixels=MAX_PIXELS)

    def run(self, img, prompt):
        msgs = [{"role": "user", "content": [
            {"type": "image", "image": img},
            {"type": "text", "text": prompt}]}]
        text = self.processor.apply_chat_template(
            msgs, tokenize=False, add_generation_prompt=True)
        imgs, vids = process_vision_info(msgs)
        inp = self.processor(text=[text], images=imgs, videos=vids,
                             padding=True, return_tensors="pt").to(self.model.device)
        with torch.inference_mode():
            gen = self.model.generate(**inp, max_new_tokens=MAX_NEW_TOKENS,
                                      do_sample=False, return_dict_in_generate=True,
                                      output_scores=True)
        trimmed = [o[len(i):] for i, o in zip(inp.input_ids, gen.sequences)]
        raw = self.processor.batch_decode(trimmed, skip_special_tokens=True)[0].strip()
        conf = sequence_confidence(self.model, gen.sequences, gen.scores)
        return raw, conf


class SmolVLM:
    """The non-dominant screening model. Idefics3-style API: no
    qwen_vl_utils, images are passed to the processor directly."""

    def __init__(self):
        print("Loading SMALL", SMALL_MODEL_ID, "->", SMALL_DEVICE, "...", flush=True)
        try:
            self.model = AutoVLM.from_pretrained(
                SMALL_MODEL_ID, attn_implementation="sdpa", **{_DTYPE_KW: DTYPE})
        except Exception as e:
            print("  sdpa unavailable (%s); falling back to eager" % type(e).__name__)
            self.model = AutoVLM.from_pretrained(SMALL_MODEL_ID, **{_DTYPE_KW: DTYPE})
        self.model.to(SMALL_DEVICE).eval()
        self.processor = AutoProcessor.from_pretrained(
            SMALL_MODEL_ID, size={"longest_edge": SMALL_LONGEST_EDGE})

    def run(self, img, prompt):
        msgs = [{"role": "user", "content": [
            {"type": "image"}, {"type": "text", "text": prompt}]}]
        text = self.processor.apply_chat_template(msgs, add_generation_prompt=True)
        inp = self.processor(text=text, images=[img], return_tensors="pt")
        inp = {k: (v.to(SMALL_DEVICE) if hasattr(v, "to") else v) for k, v in inp.items()}
        with torch.inference_mode():
            gen = self.model.generate(**inp, max_new_tokens=SMALL_MAX_NEW_TOKENS,
                                      do_sample=False, return_dict_in_generate=True,
                                      output_scores=True)
        in_len = inp["input_ids"].shape[1]
        raw = self.processor.batch_decode(
            gen.sequences[:, in_len:], skip_special_tokens=True)[0].strip()
        conf = sequence_confidence(self.model, gen.sequences, gen.scores)
        return raw, conf


LARGE = QwenVLM() if MODE in {"cascade", "large_only"} else None
SMALL = SmolVLM() if MODE in {"cascade", "small_only"} else None
print("Ready | mode=%s | conf_threshold=%.2f | exam context=%s"
      % (MODE, CONF_THRESHOLD, LANG), flush=True)

# ------------------------------------------------------------------ stats ---

STATS = {"n": 0, "n_fast": 0, "n_escalated": 0,
         "sum_small_ms": 0.0, "sum_large_ms": 0.0,
         "sum_total_ms": 0.0, "sum_fast_ms": 0.0, "sum_esc_ms": 0.0,
         "n_small_flag": 0, "n_final_flag": 0, "n_overruled": 0,
         "n_parse_fail_small": 0}


def _record(path, t_small, t_large, t_total, small_flag, final_flag,
            overruled, parse_fail_small):
    with _STATS_LOCK:
        STATS["n"] += 1
        STATS["sum_small_ms"] += t_small or 0.0
        STATS["sum_large_ms"] += t_large or 0.0
        STATS["sum_total_ms"] += t_total
        if path == "fast":
            STATS["n_fast"] += 1
            STATS["sum_fast_ms"] += t_total
        elif path == "escalated":
            STATS["n_escalated"] += 1
            STATS["sum_esc_ms"] += t_total
        STATS["n_small_flag"] += int(bool(small_flag))
        STATS["n_final_flag"] += int(bool(final_flag))
        STATS["n_overruled"] += int(bool(overruled))
        STATS["n_parse_fail_small"] += int(bool(parse_fail_small))


# ---------------------------------------------------------------- analyze ---

def _err(msg):
    return {"category": "error", "identity": "unknown", "exam_relevance": "none",
            "summary": msg, "is_cheating": False, "latency_ms": None}


def _run_guarded(runner, img, prompt, tag):
    """Run one model under the global lock.

    Returns (raw, conf, queue_ms, generate_ms, error). The two timings are kept
    apart on purpose: when several screenshots arrive at once they serialise on
    _GEN_LOCK, and folding that wait into "inference time" makes a queue backlog
    look like a slow model. Report generate_ms as inference; queue_ms belongs to
    throughput, not to the model.
    """
    t_enter = time.perf_counter()
    with _GEN_LOCK:
        t_start = time.perf_counter()
        queue_ms = (t_start - t_enter) * 1000.0
        try:
            raw, conf = runner.run(img, prompt)
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            return None, 0.0, queue_ms, None, (
                "%s: GPU out of memory; lower MAX_PIXELS, the capture "
                "resolution, or use LARGE_LOAD_4BIT=1" % tag)
        except Exception as e:
            return None, 0.0, queue_ms, None, "%s inference failed: %s: %s" % (
                tag, type(e).__name__, e)
        finally:
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
    return raw, conf, queue_ms, (time.perf_counter() - t_start) * 1000.0, None


def analyze(img):
    t_wall = time.perf_counter()
    t_small = t_large = None
    small_queue = small_gen = large_queue = large_gen = None
    small_out = None
    small_conf = 0.0
    small_flag = None
    parse_ok_small = True
    small_valid = True
    small_invalid_reason = None
    small_warning = None

    # -- stage 1: screening by the non-dominant model ------------------------
    if MODE in {"cascade", "small_only"}:
        raw, small_conf, small_queue, small_gen, err = _run_guarded(
            SMALL, img, build_prompt(), "small")
        if err:
            return _err(err)
        t_small = small_queue + small_gen
        small_out, parse_ok_small = parse_json(raw)
        small_out.setdefault("identity", "unknown")
        small_out.setdefault("summary", "")
        small_flag, small_warning = decide(small_out)
        small_valid, small_invalid_reason = output_valid(small_out)

    if MODE == "small_only":
        result = dict(small_out)
        is_cheating, warning = decide(result)
        total_ms = (time.perf_counter() - t_wall) * 1000.0
        result.update({"is_cheating": is_cheating, "latency_ms": round(total_ms),
                       "path": "small_only", "escalated": False,
                       "small": {"category": small_out.get("category"),
                                 "exam_relevance": small_out.get("exam_relevance"),
                                 "is_cheating": small_flag,
                                 "schema_valid": small_valid,
                                 "confidence": round(small_conf, 3),
                                 "queue_ms": round(small_queue),
                                 "generate_ms": round(small_gen),
                                 "latency_ms": round(t_small)}})
        if warning:
            result["warning"] = warning
        _record("small_only", t_small, None, total_ms, small_flag, is_cheating,
                False, (not parse_ok_small) or (not small_valid))
        _log(result)
        return result

    # -- gate: clear the fast path only when the screen is plainly the exam ---
    #
    # The gate deliberately does NOT key off the small model's cheating verdict.
    # That verdict is False in three very different situations: the screen
    # really is just the exam; the model echoed the prompt template instead of
    # answering; or the model saw ChatGPT but judged exam_relevance as "none".
    # Only the first is evidence of innocence. Keying off "not cheating" made
    # the cascade skip precisely the screenshots it exists to catch.
    #
    # So the small model is trusted for exactly one claim -- "there is nothing
    # here but the exam page" -- and everything else escalates. Relevance is the
    # hard judgment; it belongs to the dominant model.
    if MODE == "cascade":
        clean = (str(small_out.get("category") or "").strip().lower()
                 == "exam_page") and parse_ok_small and small_valid
        confident = small_conf >= CONF_THRESHOLD
        if clean and confident:
            total_ms = (time.perf_counter() - t_wall) * 1000.0
            result = {
                "category": small_out.get("category", "exam_page"),
                "identity": small_out.get("identity", "unknown"),
                "exam_relevance": small_out.get("exam_relevance", "none"),
                "summary": small_out.get("summary", ""),
                "is_cheating": False,
                "latency_ms": round(total_ms),
                "path": "fast",
                "escalated": False,
                "gate": {"clean": clean, "confident": confident,
                         "confidence": round(small_conf, 3),
                         "threshold": CONF_THRESHOLD,
                         "rule": "category == exam_page"},
                "small": {"category": small_out.get("category"),
                          "exam_relevance": small_out.get("exam_relevance"),
                          "is_cheating": False,
                          "schema_valid": small_valid,
                          "confidence": round(small_conf, 3),
                          "queue_ms": round(small_queue),
                          "generate_ms": round(small_gen),
                          "latency_ms": round(t_small)},
                "large": None,
            }
            if small_warning:
                result["warning"] = small_warning
            _record("fast", t_small, None, total_ms, False, False, False,
                    (not parse_ok_small) or (not small_valid))
            _log(result)
            return result

    # -- stage 2: the dominant model decides ---------------------------------
    prompt = (build_escalation_prompt(small_out, small_conf)
              if MODE == "cascade" and small_out else build_prompt())
    raw, large_conf, large_queue, large_gen, err = _run_guarded(LARGE, img, prompt, "large")
    if err:
        return _err(err)
    t_large = large_queue + large_gen
    large_out, parse_ok_large = parse_json(raw)
    large_out.setdefault("identity", "unknown")
    large_out.setdefault("summary", "")
    is_cheating, warning = decide(large_out)

    total_ms = (time.perf_counter() - t_wall) * 1000.0
    overruled = (MODE == "cascade" and small_flag is not None
                 and bool(small_flag) != bool(is_cheating))

    result = dict(large_out)
    result.update({
        "is_cheating": is_cheating,
        "latency_ms": round(total_ms),
        "path": "escalated" if MODE == "cascade" else "large_only",
        "escalated": MODE == "cascade",
        "large": {"category": large_out.get("category"),
                  "exam_relevance": large_out.get("exam_relevance"),
                  "is_cheating": is_cheating,
                  "confidence": round(large_conf, 3),
                  "parsed_ok": parse_ok_large,
                  "queue_ms": round(large_queue),
                  "generate_ms": round(large_gen),
                  "latency_ms": round(t_large)},
    })
    if MODE == "cascade":
        small_cat = str(small_out.get("category") or "").strip().lower()
        result["gate"] = {"clean": small_cat == "exam_page" and parse_ok_small
                                   and small_valid,
                          "confident": small_conf >= CONF_THRESHOLD,
                          "confidence": round(small_conf, 3),
                          "threshold": CONF_THRESHOLD,
                          "rule": "category == exam_page",
                          "escalation_reason": (
                              small_invalid_reason if not small_valid
                              else "json parse failed" if not parse_ok_small
                              else "confidence %.3f < %.2f" % (small_conf, CONF_THRESHOLD)
                              if small_conf < CONF_THRESHOLD
                              else "category %r is not exam_page" % small_cat)}
        result["small"] = {"category": small_out.get("category"),
                           "exam_relevance": small_out.get("exam_relevance"),
                           "is_cheating": small_flag,
                           "parsed_ok": parse_ok_small,
                           "schema_valid": small_valid,
                           "confidence": round(small_conf, 3),
                           "queue_ms": round(small_queue),
                           "generate_ms": round(small_gen),
                           "latency_ms": round(t_small)}
        result["overruled_small"] = overruled
    if warning:
        result["warning"] = warning

    _record(result["path"], t_small, t_large, total_ms, small_flag, is_cheating,
            overruled,
            ((not parse_ok_small) or (not small_valid)) if small_out else False)
    _log(result)
    return result


def _log(r):
    small = r.get("small") or {}
    gate = r.get("gate") or {}
    bits = []
    if small.get("category") is not None:
        bits.append("small=%s/%s" % (small.get("category"), small.get("exam_relevance")))
    if small.get("schema_valid") is False:
        bits.append("SMALL-UNUSABLE")
    if gate.get("escalation_reason"):
        bits.append("why=%s" % gate["escalation_reason"])
    if r.get("overruled_small"):
        bits.append("OVERRULED")
    if r.get("warning"):
        bits.append("!! " + r["warning"])
    print("[%s] %-10s %-18s rel=%-8s cheat=%-5s conf=%-5s %5dms  %s" % (
        datetime.now().strftime("%H:%M:%S"),
        r.get("path"), r.get("category"), r.get("exam_relevance"),
        r.get("is_cheating"),
        ("%.2f" % small["confidence"]) if small.get("confidence") is not None else "-",
        r.get("latency_ms") or 0, "  ".join(bits)), flush=True)


# --------------------------------------------------------------- endpoints --

app = Flask(__name__)


@app.route("/health")
def health():
    return jsonify({
        "status": "ok", "mode": MODE, "conf_threshold": CONF_THRESHOLD,
        "large_model": LARGE_MODEL_ID if LARGE else None,
        "small_model": SMALL_MODEL_ID if SMALL else None,
        "large_device": LARGE_DEVICE if LARGE else None,
        "small_device": SMALL_DEVICE if SMALL else None,
        "n_gpu": N_GPU, "lang": LANG, "exam_context": EXAM_CONTEXT})


@app.route("/config", methods=["GET", "POST"])
def config():
    global EXAM_CONTEXT, LANG, MODE, CONF_THRESHOLD
    if request.method == "POST":
        data = request.get_json(force=True, silent=True) or {}
        touched = False

        lang, ctx = data.get("lang"), data.get("exam_context")
        if lang:
            if lang not in EXAM_CONTEXTS:
                return jsonify({"error": "unknown lang %r; known: %s"
                                % (lang, sorted(EXAM_CONTEXTS))}), 400
            LANG, EXAM_CONTEXT, touched = lang, EXAM_CONTEXTS[lang], True
        elif isinstance(ctx, str) and ctx.strip():
            LANG, EXAM_CONTEXT, touched = "custom", ctx.strip(), True

        if "mode" in data:
            m = str(data["mode"])
            if m not in VALID_MODES:
                return jsonify({"error": "mode must be one of %s" % sorted(VALID_MODES)}), 400
            if m in {"cascade", "large_only"} and LARGE is None:
                return jsonify({"error": "large model not loaded; restart with MODE=%s" % m}), 409
            if m in {"cascade", "small_only"} and SMALL is None:
                return jsonify({"error": "small model not loaded; restart with MODE=%s" % m}), 409
            MODE, touched = m, True

        if "conf_threshold" in data:
            try:
                v = float(data["conf_threshold"])
            except (TypeError, ValueError):
                return jsonify({"error": "conf_threshold must be a number in [0,1]"}), 400
            if not 0.0 <= v <= 1.0:
                return jsonify({"error": "conf_threshold must be in [0,1]"}), 400
            CONF_THRESHOLD, touched = v, True

        if not touched:
            return jsonify({"error": "send lang, exam_context, mode or conf_threshold"}), 400
        print("[config] mode=%s conf_threshold=%.2f lang=%s"
              % (MODE, CONF_THRESHOLD, LANG), flush=True)
    return jsonify({"lang": LANG, "exam_context": EXAM_CONTEXT,
                    "mode": MODE, "conf_threshold": CONF_THRESHOLD})


@app.route("/analyze", methods=["POST"])
def analyze_route():
    data = request.get_json(force=True, silent=True) or {}
    if "image" not in data:
        return jsonify({"error": "missing 'image'"}), 400
    try:
        img = dataurl_to_pil(data["image"])
    except Exception as e:
        return jsonify({"error": "bad image: " + str(e)}), 400
    return jsonify(analyze(img))


@app.route("/stats")
def stats():
    with _STATS_LOCK:
        s = dict(STATS)
    n = max(s["n"], 1)
    return jsonify({
        "mode": MODE, "conf_threshold": CONF_THRESHOLD,
        "n_screenshots": s["n"],
        "n_fast_path": s["n_fast"],
        "n_escalated": s["n_escalated"],
        "escalation_rate": round(s["n_escalated"] / n, 4),
        "mean_total_ms": round(s["sum_total_ms"] / n, 1),
        "mean_fast_path_ms": round(s["sum_fast_ms"] / max(s["n_fast"], 1), 1),
        "mean_escalated_ms": round(s["sum_esc_ms"] / max(s["n_escalated"], 1), 1),
        "mean_small_ms": round(s["sum_small_ms"] / n, 1),
        "mean_large_ms": round(s["sum_large_ms"] / max(s["n_escalated"], 1), 1),
        "n_small_flagged": s["n_small_flag"],
        "n_final_flagged": s["n_final_flag"],
        "n_large_overruled_small": s["n_overruled"],
        "n_small_unusable_answers": s["n_parse_fail_small"]})


@app.route("/stats/reset", methods=["POST"])
def stats_reset():
    with _STATS_LOCK:
        for k in STATS:
            STATS[k] = 0 if isinstance(STATS[k], int) else 0.0
    return jsonify({"status": "reset"})


if __name__ == "__main__":
    print("Serving on http://%s:%d  (mode=%s)" % (HOST, PORT, MODE))
    app.run(host=HOST, port=PORT)


In [ ]:
%%writefile /kaggle/working/cv_service.py
"""Headless CV proctoring inference service for exam-monitor-extension.

Wraps the ../computer_vision project's integration modules (01_head_gaze_adapter,
02_yolo_output_adapter, 03_event_manager, 05_multi_cue_review_score) behind a
small stateful HTTP API, so inference can run on a remote GPU (e.g. a Kaggle
notebook, see kaggle/README.md) instead of the student's own machine.

This file only IMPORTS those modules by path (the same dynamic-loading trick
run_integrated_demo.py uses, since filenames starting with digits aren't valid
Python identifiers). It never modifies anything under computer_vision/ — that
directory belongs to a teammate's module.

Endpoints:
  GET  /health                                  -> service + session status
  POST /session/start  { studentId }            -> warm up a session
  POST /frame  { studentId, image, timestamp? } -> one review-score reading
  POST /session/end    { studentId }            -> release a session's resources

Run:
  python cv_service.py            # serves http://localhost:8789

Configuration is entirely via CV_* environment variables (see the block below)
because the Kaggle filesystem layout differs from this repo's local layout.

The review_score/review_level values below come straight from
05_multi_cue_review_score.py, whose own module docstring is explicit that this
is "an experimental prioritisation indicator, not a calibrated probability of
cheating, and must not be used as an automatic verdict" — keep that framing in
any downstream UI/logs; the "disclaimer" field in every /frame response is
that exact string, passed through verbatim.
"""
import base64
import importlib.util
import os
import sys
import threading
import time
from pathlib import Path
from types import ModuleType
from typing import Any, Dict

import cv2
import numpy as np
from flask import Flask, request, jsonify

EXT_DIR = Path(__file__).resolve().parent
CV_ROOT = EXT_DIR.parent / "computer_vision"
# Overridable because a Kaggle notebook lays the integration modules out flat
# (from an attached Dataset) rather than in this repo's computer_vision/src/
# integration/ layout.
INTEGRATION_DIR = Path(os.environ.get("CV_INTEGRATION_DIR") or (CV_ROOT / "src" / "integration"))

PORT = int(os.environ.get("CV_PORT", "8789"))
IDLE_SESSION_TIMEOUT_S = float(os.environ.get("CV_IDLE_SESSION_TIMEOUT_S", "600"))

L2CS_ROOT = Path(os.environ.get("CV_L2CS_ROOT") or (CV_ROOT / "external" / "L2CS-Net"))
L2CS_SNAPSHOT = Path(os.environ.get("CV_L2CS_SNAPSHOT") or (CV_ROOT / "models" / "l2cs" / "L2CSNet_gaze360.pkl"))
MEDIAPIPE_MODEL = Path(os.environ.get("CV_MEDIAPIPE_MODEL") or (CV_ROOT / "models" / "mediapipe" / "face_landmarker.task"))
CANONICAL14_CSV = Path(os.environ.get("CV_CANONICAL14_CSV") or (CV_ROOT / "resources" / "mediapipe" / "mediapipe_expanded_subset_14.csv"))
YOLO_MODEL_DIR = Path(os.environ.get("CV_YOLO_MODEL_DIR") or (CV_ROOT / "models" / "yolo"))
YOLO_CHECKPOINT = os.environ.get("CV_YOLO_CHECKPOINT", "5e")
REVIEW_CONFIG_PATH = Path(os.environ.get("CV_REVIEW_CONFIG") or (CV_ROOT / "configs" / "multi_cue_review_score_v1_1.json"))


def _load_local_module(path: Path, module_name: str) -> ModuleType:
    """Import a numbered integration module by file path (identifiers can't start with 01/02/...)."""
    path = Path(path).resolve()
    if not path.is_file():
        raise FileNotFoundError(f"Required integration module not found: {path}")
    spec = importlib.util.spec_from_file_location(module_name, path)
    if spec is None or spec.loader is None:
        raise ImportError(f"Could not create import spec for: {path}")
    module = importlib.util.module_from_spec(spec)
    sys.modules[module_name] = module
    spec.loader.exec_module(module)
    return module


def _resolve_device() -> str:
    env_device = os.environ.get("CV_DEVICE")
    if env_device:
        # An explicit CV_DEVICE=cuda is trusted at face value here, but NOT
        # left unvalidated -- see the torch.cuda.is_available() check right
        # after this function is called below. Without that check, a stale
        # CUDA_VISIBLE_DEVICES or a pip install that clobbered the CUDA-linked
        # torch build (e.g. an unpinned transitive dependency) only surfaces
        # as a confusing failure deep inside the first /frame request
        # ("Invalid CUDA 'device=0' requested") instead of a clear boot error.
        return env_device
    try:
        import torch
        return "cuda" if torch.cuda.is_available() else "cpu"
    except Exception:
        return "cpu"


def _adapter_device_string(device: str) -> str:
    """Normalize a device string for the adapters below.

    01_head_gaze_adapter.py calls l2cs's own select_device(), an
    older YOLOv5-vintage helper that only understands '', 'cpu', an index
    ('0'), or a comma list ('0,1') -- NOT a bare 'cuda' with no index. Given
    'cuda', that helper ends up doing `os.environ['CUDA_VISIBLE_DEVICES'] =
    'cuda'` (a non-numeric value), which the CUDA driver then rejects with
    "Invalid device id". A bare 'cuda' is normalized to '0' (first visible
    GPU) here; an explicit index, 'cuda:0', or 'cpu' passes through
    unchanged. Applied to both adapters' device config for consistency, even
    though Ultralytics YOLO's own device parsing already tolerates 'cuda'.
    """
    return "0" if device.strip().lower() == "cuda" else device


def _require_paths(*paths: Path) -> None:
    missing = [str(p) for p in paths if not p.exists()]
    if missing:
        print("cv_service: missing required CV assets:")
        for m in missing:
            print("  -", m)
        print(
            "See computer_vision/models/README.md for local setup, or "
            "exam-monitor-extension/kaggle/README.md for the Kaggle layout, "
            "and set the CV_* environment variables if assets live elsewhere."
        )
        sys.exit(1)


print("cv_service: loading integration modules from", INTEGRATION_DIR)
head_module = _load_local_module(INTEGRATION_DIR / "01_head_gaze_adapter.py", "teep_head_gaze_adapter")
yolo_module = _load_local_module(INTEGRATION_DIR / "02_yolo_output_adapter.py", "teep_yolo_output_adapter")
event_module = _load_local_module(INTEGRATION_DIR / "03_event_manager.py", "teep_event_manager")
review_module = _load_local_module(INTEGRATION_DIR / "05_multi_cue_review_score.py", "teep_multi_cue_review_score")

DEVICE = _resolve_device()

if DEVICE == "cuda":
    try:
        import torch
        cuda_ok = torch.cuda.is_available()
    except Exception as e:
        cuda_ok = False
        print(f"cv_service: torch import/CUDA check failed: {e}")
    if not cuda_ok:
        print(
            "cv_service: CV_DEVICE=cuda was requested but "
            "torch.cuda.is_available() is False -- refusing to start with a "
            "device that will fail on the first real request instead of here.\n"
            "Common causes on Kaggle: GPU accelerator not enabled for this "
            "session, or a pip install (e.g. an unpinned transitive "
            "dependency) silently replaced the preinstalled CUDA-linked "
            "torch build with a CPU-only one -- try restarting the kernel "
            "and re-running the install cell with --no-deps where possible.\n"
            "Set CV_DEVICE=cpu to run on CPU instead (slow) once you've "
            "confirmed that's actually what you want."
        )
        sys.exit(1)

_require_paths(L2CS_ROOT, L2CS_SNAPSHOT, MEDIAPIPE_MODEL, CANONICAL14_CSV, REVIEW_CONFIG_PATH)

print(f"cv_service: device={DEVICE} | yolo_checkpoint={YOLO_CHECKPOINT}")

REVIEW_SCORE_CONFIG = review_module.load_review_score_config(REVIEW_CONFIG_PATH)

# Stateless per frame (no calibration, no temporal state) -> one shared instance
# is safe across every student session and avoids reloading the checkpoint per student.
print("cv_service: loading YOLO checkpoint...")
_yolo_adapter = yolo_module.YoloOutputAdapter(
    config=yolo_module.YoloAdapterConfig(checkpoint_id=YOLO_CHECKPOINT, device=_adapter_device_string(DEVICE)),
    model_dir=YOLO_MODEL_DIR,
)
_yolo_adapter.start()

# Frozen defaults (README-documented event thresholds) — not overridden here.
_EVENT_RULES = event_module.build_initial_integration_rules(enable_audio_device_event=True)
_OBJECT_EVENT_ELIGIBILITY = event_module.build_initial_object_event_eligibility()


class _Session:
    """Per-studentId state. HeadGazeAdapter/EventManager/MultiCueReviewScorer are
    all stateful (calibration, temporal cue tracking, session-peak score), so
    each student needs their own instance — unlike the shared YOLO adapter above.
    """

    __slots__ = ("student_id", "head_adapter", "event_manager", "review_scorer", "last_seen", "frame_count")

    def __init__(self, student_id: str):
        self.student_id = student_id
        self.head_adapter = head_module.HeadGazeAdapter(
            l2cs_root=L2CS_ROOT,
            l2cs_snapshot=L2CS_SNAPSHOT,
            mediapipe_model=MEDIAPIPE_MODEL,
            canonical14_csv=CANONICAL14_CSV,
            config=head_module.HeadGazeConfig(device=_adapter_device_string(DEVICE), mirror_input=False),
        )
        self.head_adapter.start()
        self.event_manager = event_module.EventManager(
            _EVENT_RULES,
            object_event_eligibility=_OBJECT_EVENT_ELIGIBILITY,
            suppress_no_person_if_face_present=True,
        )
        self.review_scorer = review_module.MultiCueReviewScorer(config=REVIEW_SCORE_CONFIG)
        self.last_seen = time.time()
        self.frame_count = 0

    def close(self) -> None:
        try:
            self.head_adapter.close()
        except Exception as e:
            print(f"cv_service: error closing session {self.student_id!r}: {e}")


_sessions: Dict[str, _Session] = {}
_sessions_lock = threading.Lock()
# One Kaggle GPU behind this service -> serialize every model call across
# sessions, same reasoning vlm_service.py already applies with its own lock.
_inference_lock = threading.Lock()


def _get_or_create_session(student_id: str) -> _Session:
    with _sessions_lock:
        session = _sessions.get(student_id)
        if session is not None:
            return session

    # Cold-starting a session calls HeadGazeAdapter.start(), which loads L2CS +
    # MediaPipe onto the GPU. Serialize that against in-flight /frame inference
    # through the same _inference_lock _process_frame uses below -- one Kaggle
    # GPU, avoid a new session's model load racing an active inference call
    # for CUDA memory (this raced uninstrumented before and could crash the
    # request with an unhandled exception -- see the try/except in the routes
    # that call this function).
    with _inference_lock:
        with _sessions_lock:
            session = _sessions.get(student_id)
            if session is not None:
                return session  # created by another thread while we waited
        print(f"cv_service: starting session for studentId={student_id!r}")
        session = _Session(student_id)
        with _sessions_lock:
            _sessions[student_id] = session
        return session


def _drop_session(student_id: str) -> bool:
    with _sessions_lock:
        session = _sessions.pop(student_id, None)
    if session is None:
        return False
    session.close()
    return True


def _evict_idle_sessions_loop() -> None:
    # No "press Q to quit" teardown here (unlike the desktop demo), so idle
    # sessions must be reaped or each one leaks a worker thread + landmarker.
    while True:
        time.sleep(60.0)
        now = time.time()
        with _sessions_lock:
            stale_ids = [sid for sid, s in _sessions.items() if now - s.last_seen > IDLE_SESSION_TIMEOUT_S]
        for sid in stale_ids:
            print(f"cv_service: evicting idle session studentId={sid!r}")
            _drop_session(sid)


threading.Thread(target=_evict_idle_sessions_loop, name="cv-session-reaper", daemon=True).start()


def _decode_frame(image_field: str) -> np.ndarray:
    s = image_field
    if isinstance(s, str) and s.strip().startswith("data:") and "," in s:
        s = s.split(",", 1)[1]
    raw = base64.b64decode(s)
    frame = cv2.imdecode(np.frombuffer(raw, dtype=np.uint8), cv2.IMREAD_COLOR)
    if frame is None:
        raise ValueError("Could not decode image as JPEG/PNG.")
    return frame


def _process_frame(session: _Session, frame: np.ndarray, timestamp: float) -> Dict[str, Any]:
    with _inference_lock:
        head_output = session.head_adapter.process_frame(frame)
        calibration = head_output.get("calibration") or {}
        if calibration.get("head_confirmation_required"):
            # Scripted equivalent of the desktop demo's "press C" step: as soon
            # as a stable neutral baseline is ready, confirm it automatically —
            # there's no interactive user here to press a key.
            session.head_adapter.confirm_head_baseline()
            calibration = session.head_adapter.calibration_status()

        yolo_output = _yolo_adapter.process_frame(frame, timestamp=timestamp)
        event_output = session.event_manager.process(head_output, yolo_output, timestamp=timestamp)
        review_output = session.review_scorer.process(event_output)

    session.frame_count += 1
    session.last_seen = time.time()

    return {
        # Explicit float(): 05_multi_cue_review_score.py may hand back a
        # numpy scalar depending on how it computed the value internally, and
        # jsonify() raises (uncaught, since this happens during response
        # serialization *after* the route's own try/except already
        # succeeded) if it hits one of those instead of a native float.
        "review_score": float(review_output["score"]),
        "review_level": review_output["review_level"],
        "session_peak_score": float(review_output["session_peak_score"]),
        "active_cues": list(review_output["active_cues"]),
        "calibration_phase": calibration.get("phase"),
        "frame_count": int(session.frame_count),
        "disclaimer": review_output["disclaimer"],
        # Raw per-frame YOLO detections (already confidence-thresholded by
        # 02_yolo_output_adapter.py itself) so the dashboard can draw
        # bounding boxes over the webcam thumbnail. bbox_xyxy is in the pixel
        # space of the frame the extension captured (currently 320x240) --
        # the dashboard reads the displayed <img>'s natural size rather than
        # assuming that resolution, so this stays correct if it ever changes.
        "detections": [
            {
                "label": str(d.get("label", "unknown")),
                "confidence": float(d.get("confidence", 0.0)),
                "bbox_xyxy": [float(v) for v in d.get("bbox_xyxy", [0, 0, 0, 0])],
            }
            for d in (yolo_output.get("detections") or [])
        ],
    }


app = Flask(__name__)


@app.errorhandler(Exception)
def _handle_unexpected_error(e):
    # Global safety net: ANY unhandled exception anywhere in a route (not
    # just the ones explicitly try/except'd below) must still come back as
    # JSON, never Flask/Werkzeug's default HTML error page -- server.js's
    # analyzeWithCV() does response.json() and has no HTML fallback, so an
    # HTML response there previously surfaced as an opaque "Unexpected
    # token '<'" instead of the actual error.
    import traceback
    print("cv_service: unhandled exception:")
    traceback.print_exc()
    return jsonify({"error": "unhandled server error: " + str(e)}), 500


@app.route("/health")
def health():
    with _sessions_lock:
        active = len(_sessions)
    return jsonify({
        "status": "ok",
        "device": DEVICE,
        "yolo_checkpoint": YOLO_CHECKPOINT,
        "sessions_active": active,
    })


@app.route("/session/start", methods=["POST"])
def session_start():
    data = request.get_json(force=True, silent=True) or {}
    student_id = str(data.get("studentId") or "").strip()
    if not student_id:
        return jsonify({"error": "missing 'studentId'"}), 400
    try:
        session = _get_or_create_session(student_id)
    except Exception as e:
        print(f"cv_service: /session/start error for studentId={student_id!r}: {e}")
        return jsonify({"error": "session start failed: " + str(e)}), 500
    return jsonify({"ok": True, "calibration_phase": session.head_adapter.calibration_status().get("phase")})


@app.route("/frame", methods=["POST"])
def frame_route():
    data = request.get_json(force=True, silent=True) or {}
    student_id = str(data.get("studentId") or "").strip()
    if not student_id:
        return jsonify({"error": "missing 'studentId'"}), 400
    if "image" not in data:
        return jsonify({"error": "missing 'image'"}), 400

    try:
        decoded_frame = _decode_frame(data["image"])
    except Exception as e:
        return jsonify({"error": "bad image: " + str(e)}), 400

    timestamp = data.get("timestamp")
    # Client sends epoch milliseconds (JS Date.now()); event durations only
    # need monotonically increasing seconds, so epoch seconds works directly.
    ts = float(timestamp) / 1000.0 if isinstance(timestamp, (int, float)) else time.time()

    try:
        session = _get_or_create_session(student_id)
    except Exception as e:
        print(f"cv_service: /frame session-start error for studentId={student_id!r}: {e}")
        return jsonify({"error": "session start failed: " + str(e)}), 500

    try:
        result = _process_frame(session, decoded_frame, ts)
    except Exception as e:
        print(f"cv_service: /frame error for studentId={student_id!r}: {e}")
        return jsonify({"error": "inference failed: " + str(e)}), 500

    return jsonify(result)


@app.route("/session/end", methods=["POST"])
def session_end():
    data = request.get_json(force=True, silent=True) or {}
    student_id = str(data.get("studentId") or "").strip()
    if not student_id:
        return jsonify({"error": "missing 'studentId'"}), 400
    dropped = _drop_session(student_id)
    return jsonify({"ok": True, "dropped": dropped})


if __name__ == "__main__":
    print(f"cv_service: serving on http://0.0.0.0:{PORT}")
    app.run(host="0.0.0.0", port=PORT, threaded=True)


## 5. Configure the cascade

GPU placement for T4 x2 with all three services running:

| Component | Device | Rough VRAM |
|---|---|---|
| Qwen2.5-VL-3B (large) | `cuda:0` | ~7 GB |
| CV stack (YOLO + L2CS + MediaPipe) | `cuda:0` | ~2 GB |
| SmolVLM-500M (small) | `cuda:1` | ~1 GB |

**To try the 7B here anyway**, set `LARGE_MODEL_ID` to the 7B *and*
`LARGE_DEVICE = "auto"` with `LARGE_MAX_MEMORY = "0:9GiB,1:9GiB"` — that shards
it across both GPUs while leaving room for the other two components. Expect it
to be tight; if it OOMs, run the CV service in a separate session or drop back
to the 3B.

In [ ]:
import os, torch

MODE = "cascade"            # "cascade" | "large_only" | "small_only"
CONF_THRESHOLD = "0.60"
EXAM_LANG = "en"            # en | ko | zh | ja

LARGE_MODEL_ID = "Qwen/Qwen2.5-VL-3B-Instruct"      # see the note above re 7B
SMALL_MODEL_ID = "HuggingFaceTB/SmolVLM-500M-Instruct"
LARGE_DEVICE = "cuda:0"     # or "auto" to shard across both GPUs (needed for 7B)
SMALL_DEVICE = "cuda:1" if torch.cuda.device_count() > 1 else "cuda:0"
LARGE_MAX_MEMORY = ""       # e.g. "0:9GiB,1:9GiB", only used when LARGE_DEVICE="auto"

if torch.cuda.device_count() < 2:
    print("WARNING: only {} GPU visible. Three models on one T4 is very likely "
          "to OOM. Switch the accelerator to T4 x2.".format(torch.cuda.device_count()))

vlm_env = os.environ.copy()
vlm_env.update({
    "MODE": MODE,
    "PORT": "8791",
    "HOST": "0.0.0.0",
    "REQUIRE_CUDA": "1",
    "CONF_THRESHOLD": CONF_THRESHOLD,
    "EXAM_LANG": EXAM_LANG,
    "LARGE_MODEL_ID": LARGE_MODEL_ID,
    "SMALL_MODEL_ID": SMALL_MODEL_ID,
    "LARGE_DEVICE": LARGE_DEVICE,
    "SMALL_DEVICE": SMALL_DEVICE,
})
if LARGE_MAX_MEMORY:
    vlm_env["LARGE_MAX_MEMORY"] = LARGE_MAX_MEMORY

print("mode=%s  threshold=%s  lang=%s" % (MODE, CONF_THRESHOLD, EXAM_LANG))
print("large=%s on %s" % (LARGE_MODEL_ID, LARGE_DEVICE))
print("small=%s on %s" % (SMALL_MODEL_ID, SMALL_DEVICE))


## 6. Start both services in the background

`cv_service.py`'s `CV_*` paths point at the attached dataset; the cascade takes
its configuration from the cell above.

In [ ]:
import subprocess, sys

cv_env = os.environ.copy()
cv_env.update({
    "CV_DEVICE": "cuda",
    "CV_INTEGRATION_DIR": str(DATASET_DIR / "integration"),
    "CV_REVIEW_CONFIG": str(DATASET_DIR / "configs" / "multi_cue_review_score_v1_1.json"),
    "CV_CANONICAL14_CSV": str(DATASET_DIR / "resources" / "mediapipe" / "mediapipe_expanded_subset_14.csv"),
    "CV_YOLO_MODEL_DIR": str(DATASET_DIR / "models"),  # flat layout — file sits directly under models/
    "CV_L2CS_ROOT": "/kaggle/working/L2CS-Net",
    "CV_L2CS_SNAPSHOT": str(DATASET_DIR / "models" / "L2CSNet_gaze360.pkl"),
    "CV_MEDIAPIPE_MODEL": str(DATASET_DIR / "models" / "face_landmarker.task"),
})

vlm_log = open("/kaggle/working/vlm_cascade.log", "w")
cv_log = open("/kaggle/working/cv_service.log", "w")

vlm_proc = subprocess.Popen([sys.executable, "/kaggle/working/vlm_service_cascade.py"],
                            stdout=vlm_log, stderr=subprocess.STDOUT, env=vlm_env)
cv_proc = subprocess.Popen([sys.executable, "/kaggle/working/cv_service.py"],
                           stdout=cv_log, stderr=subprocess.STDOUT, env=cv_env)

print("vlm_service_cascade.py pid:", vlm_proc.pid)
print("cv_service.py           pid:", cv_proc.pid)
print("Loading models — several minutes the first time (two VLMs download from "
      "Hugging Face, plus YOLO/L2CS/MediaPipe load). Check with the next cell.")


In [ ]:
!echo '--- vlm_cascade.log (tail) ---'; tail -n 25 /kaggle/working/vlm_cascade.log
!echo '--- cv_service.log (tail) ---'; tail -n 25 /kaggle/working/cv_service.log


Re-run the cell above until both logs show their "serving on ..." line (or an
error to fix). A dead process is easier to spot than a slow one:

In [ ]:
print("vlm alive?", vlm_proc.poll() is None, "| cv alive?", cv_proc.poll() is None)


In [ ]:
!curl -s -m 10 http://localhost:8791/health && echo
!curl -s -m 10 http://localhost:8789/health && echo


## 7. Expose both services over `cloudflared` tunnels

Two separate quick tunnels — one per port — same pattern as the single-model
notebook, with the VLM tunnel pointed at `8791` instead of `8788`.

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /kaggle/working/cloudflared
!chmod +x /kaggle/working/cloudflared

get_ipython().system_raw(
    '/kaggle/working/cloudflared tunnel --url http://localhost:8791 > /kaggle/working/tunnel_vlm.log 2>&1 &'
)
get_ipython().system_raw(
    '/kaggle/working/cloudflared tunnel --url http://localhost:8789 > /kaggle/working/tunnel_cv.log 2>&1 &'
)
print("Tunnels starting — wait ~10-15s, then run the next cell.")


In [ ]:
import re, time

time.sleep(12)

def extract_url(log_path):
    text = Path(log_path).read_text(errors="ignore")
    m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", text)
    return m.group(0) if m else None

vlm_url = extract_url("/kaggle/working/tunnel_vlm.log")
cv_url = extract_url("/kaggle/working/tunnel_cv.log")

if not vlm_url or not cv_url:
    print("Tunnel URL not found yet — re-run this cell in a few seconds.")
    print("VLM_URL:", vlm_url)
    print("CV_URL: ", cv_url)
else:
    print("Both tunnels are up. On the machine running server.js:\n")
    print(f'VLM_URL={vlm_url} CV_URL={cv_url} node server.js')


## Known operational limitations

- **Kaggle free-GPU quota** is roughly 30 GPU-hours/week per account, and this
  notebook now loads three models per session instead of two.
- **Notebook sessions time out** (idle timeout + a hard runtime cap) — both
  services and both tunnels die with the notebook.
- **Tunnel URLs change every restart.** Re-run section 7 and update
  `VLM_URL`/`CV_URL` on the `server.js` machine each time.
- **No auth on either service's endpoints** — fine for a research demo on a
  throwaway URL, not fine beyond that without at least a shared-secret header.
- **A 502 from the tunnel while the log still says `Loading LARGE ...` is
  normal** — the tunnel comes up instantly, the models take minutes. Wait, then
  retry.
- **The cascade's fast path can produce false negatives**: a screenshot the
  small model confidently clears never reaches the large model. That trade-off
  is the point of the comparison — report cascade recall next to `large_only`
  recall, not accuracy alone.